In [ ]:
"""
Concepts Covered:

1. Binary Classification
2. Probability Prediction
3. Threshold Tuning
4. Precision
5. Recall
6. F1 Score
7. ROC AUC
8. PR Curve
9. Regularization
10. Calibration
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV,learning_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,
    roc_auc_score,roc_curve,precision_recall_curve,average_precision_score)

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target
print(df.head())
print(df.info())
print(df.describe())
print("\nClass Distribution")
print(df["target"].value_counts())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())
print(df["target"].value_counts(normalize=True))

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("target", axis=1)
y = df["target"]
X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42)

In [ ]:
# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("model",LogisticRegression(penalty="l2",C=1.0,solver="lbfgs",max_iter=1000,random_state=42))])

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

pipe.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS with base model
# =====================================================

y_pred = pipe.predict(X_val)
y_prob = pipe.predict_proba(X_val)[:,1]
accuracy = accuracy_score(y_val,y_pred)
precision = precision_score(y_val,y_pred)
recall = recall_score(y_val,y_pred)
f1 = f1_score(y_val,y_pred)
roc_auc = roc_auc_score(y_val,y_pred)

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(pipe,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
print("\nCV ROC AUC")
print(cv_scores.mean())
print(cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "model__C":[0.001,0.01,0.1,1,10,100],
    "model__penalty":["l1","l2"],
    "model__solver":["liblinear"]}

grid = GridSearchCV(pipe,param_grid,cv=5,scoring="roc_auc",n_jobs=-1)
grid.fit(X_train,y_train)

print("\nBest Params")
print(grid.best_params_)
best_model = grid.best_estimator_

# =====================================================
# STEP 8.5 : RANDOMIZED SEARCH
# =====================================================

random_search = RandomizedSearchCV(pipe,param_distributions=param_grid,cv=5,n_iter=10,scoring="roc_auc",random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
print(random_search.best_params_)

In [ ]:
# =====================================================
# STEP 9 : VALIDATION with best model
# =====================================================

y_best = best_model.predict(X_val)
val_prob = best_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val,val_prob)
print("\nValidation ROC AUC")
print(val_auc)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

y_train_prob = best_model.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train,y_train_prob)
print("\nTrain AUC:", train_auc)
print("Validation AUC:", val_auc)

In [ ]:
# =====================================================
# STEP 11 : LOGISTIC REGRESSION ANALYSIS
# =====================================================

# ------------------------------------------
# Confusion Matrix
# ------------------------------------------

cm = confusion_matrix(y_val,y_best)
sns.heatmap(cm,annot=True,fmt='d')
plt.title("Confusion Matrix")
plt.show()

# ------------------------------------------
# Classification Report
# ------------------------------------------

print(classification_report(y_val,y_best))

# ------------------------------------------
# ROC Curve
# ------------------------------------------

fpr, tpr, thresholds = roc_curve(y_val,val_prob)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1])
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.show()

# ------------------------------------------
# Precision Recall Curve
# ------------------------------------------

precision_vals, recall_vals, thresholds = (precision_recall_curve(y_val,val_prob))
plt.plot(recall_vals,precision_vals)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision Recall Curve")
plt.show()

# ------------------------------------------
# Feature Importance
# ------------------------------------------

coef_df = pd.DataFrame({"Feature":X.columns,"Coefficient":best_model.named_steps["model"].coef_[0]})
coef_df["Abs_Coeff"] = np.abs(coef_df["Coefficient"])
coef_df = coef_df.sort_values(by="Abs_Coeff",ascending=False)
print(coef_df.head(20))

# ------------------------------------------
# Threshold Tuning
# ------------------------------------------

thresholds = np.arange(0.1,0.91,0.05)
scores = []

for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)
    score = f1_score(y_val,pred)
    scores.append(score)

plt.plot(thresholds,scores,marker="o")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("Threshold Tuning")
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:,1]
test_accuracy = accuracy_score(y_test,test_pred)
test_precision = precision_score(y_test,test_pred)
test_recall = recall_score(y_test,test_pred)
test_f1 = f1_score(y_test,test_pred)
test_auc = roc_auc_score(y_test,test_prob)

In [ ]:
# =====================================================
# STEP 14 : INTERPRETATION
# =====================================================

print("\nTOP FEATURES")
print(coef_df.head(15))

In [ ]:
# =====================================================
# STEP 15 : DEPLOYMENT READINESS
# =====================================================

print("\nDEPLOYMENT CHECK")
print("CV AUC:",cv_scores.mean())
print("Validation AUC:",val_auc)
print("Test AUC:",test_auc)

In [ ]:
# =====================================================
# INTERVIEW QUESTIONS
# =====================================================

"""
1. Why is Logistic Regression called regression?
2. What is sigmoid function?
3. Why do we predict probabilities?
4. What is log-odds?
5. Difference between Linear and Logistic Regression?
6. What is threshold?
7. Why is 0.5 not always optimal?
8. Accuracy vs Precision?
9. Precision vs Recall?
10. When should Recall be prioritized?
11. When should Precision be prioritized?
12. What is F1 Score?
13. What is ROC Curve?
14. What is ROC AUC?
15. What is PR Curve?
16. Why use ROC AUC for model selection?
17. What is regularization?
18. What is C parameter?
19. What happens when C increases?
20. What happens when C decreases?
21. Why scaling is important?
22. Why use Stratified Split?
23. What is class imbalance?
24. How do you handle class imbalance?
25. Explain Logistic Regression end-to-end.
"""